# 06 — Circuito publicado e activation patching semântico

Este notebook identifica as sete classes do circuito IOI publicado em Wang et al. (arXiv:2211.00593) e mede DLA e activation patching em posições semânticas por exemplo.

A lista de 26 nós vem de `Easy-Transformer/easy_transformer/ioi_circuit_extraction.py`: portanto, **não é uma descoberta do zero**. A contribuição aqui é validar e visualizar a lista publicada no dataset pareado desta replicação, mantendo Name Movers primários e Backup Name Movers separados.

In [1]:
import json
from pathlib import Path

import plotly.express as px
import torch

from ioi_replication.paper import (
    PUBLISHED_CIRCUIT,
    CircuitClass,
    SemanticPosition,
    activation_patch_matrix,
    build_paired_dataset,
    logit_difference,
    published_nodes,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED = 106
N_DEV = 4
SEMANTIC_POSITIONS = tuple(SemanticPosition)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/06_circuit_discovery.json"

device = select_device()
model = load_gpt2_small(device)
dataset = build_paired_dataset(model.tokenizer, n=N_DEV, seed=SEED)
ioi_tokens = dataset.ioi_tokens.to(device)
abc_tokens = dataset.abc_tokens.to(device)

assert len(published_nodes()) == 26
assert len(PUBLISHED_CIRCUIT) == 7
print(f"device={device}; N={N_DEV}; nós={len(published_nodes())}; classes={len(PUBLISHED_CIRCUIT)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=4; nós=26; classes=7


## DLA por posição semântica

Projetamos o `hook_result` de cada cabeça na direção de logits IO−S de cada exemplo. Em `END`, isso é uma atribuição direta ao residual que alimenta o unembedding. Nas demais posições, é uma sonda descritiva local — não afirmamos que o vetor chegue diretamente ao logit final sem processamento intermediário.

In [2]:
def metric(logits):
    return logit_difference(
        logits,
        dataset.io_token_ids.to(device),
        dataset.s_token_ids.to(device),
        dataset.positions["END"].to(device),
    )

model.set_use_attn_result(True)
try:
    ioi_logits, ioi_cache = model.run_with_cache(
        ioi_tokens,
        names_filter=lambda name: name.endswith("attn.hook_result"),
    )
    _, abc_cache = model.run_with_cache(
        abc_tokens,
        names_filter=lambda name: name.endswith("attn.hook_result"),
    )

    baseline = metric(ioi_logits).detach()
    rows = torch.arange(N_DEV, device=device)
    logit_directions = (
        model.W_U[:, dataset.io_token_ids.to(device)].T
        - model.W_U[:, dataset.s_token_ids.to(device)].T
    )
    dla = torch.empty(12, 12, len(SEMANTIC_POSITIONS), device=device)
    for layer in range(12):
        result = ioi_cache[f"blocks.{layer}.attn.hook_result"]
        for position_index, position in enumerate(SEMANTIC_POSITIONS):
            semantic_index = dataset.positions[position.value].to(device)
            selected = result[rows, semantic_index]
            dla[layer, :, position_index] = (
                selected * logit_directions[:, None, :]
            ).sum(dim=-1).mean(dim=0)
finally:
    model.set_use_attn_result(False)

dla = dla.detach()
assert dla.shape == (12, 12, len(SEMANTIC_POSITIONS))
for position_index, position in enumerate(SEMANTIC_POSITIONS):
    fig = px.imshow(
        dla[:, :, position_index].float().cpu().numpy(),
        labels={"x": "head", "y": "layer", "color": "DLA IO−S"},
        title=f"DLA de hook_result em {position.value}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="equal",
    )
    fig.update_layout(width=650, height=520)
    fig.show()
print(f"baseline IO−S: {baseline.item():.4f}")

baseline IO−S: 3.8586


## Activation patching `pABC → pIOI`

Para cada célula layer×head×posição, substituímos somente o `hook_result` daquela cabeça e a posição semântica individual de cada exemplo. O valor mostrado é a mudança em IO−S contra o forward pIOI sem patch. O lote de quatro exemplos mantém as 720 intervenções viáveis em uma RTX 3050 sem trocar a semântica do algoritmo.

In [3]:
model.set_use_attn_result(True)
try:
    patch_matrix = activation_patch_matrix(
        model=model,
        target_tokens=ioi_tokens,
        source_cache=abc_cache,
        metric=metric,
        target_positions=dataset.positions,
        source_positions=dataset.positions,
        semantic_positions=SEMANTIC_POSITIONS,
        activation="result",
    )
finally:
    model.set_use_attn_result(False)

patch_effect = patch_matrix.values - baseline
assert patch_effect.shape == (12, 12, len(SEMANTIC_POSITIONS))
position_axis = {position: i for i, position in enumerate(SEMANTIC_POSITIONS)}

for position_index, position in enumerate(SEMANTIC_POSITIONS):
    fig = px.imshow(
        patch_effect[:, :, position_index].float().cpu(),
        labels={"x": "head", "y": "layer", "color": "Δ IO−S"},
        title=f"Activation patching pABC→pIOI em {position.value}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="equal",
    )
    fig.update_layout(width=650, height=520)
    fig.show()

class_summary = {}
for circuit_class, nodes in PUBLISHED_CIRCUIT.items():
    effects = [
        patch_effect[node.layer, node.head, position_axis[node.position]].item()
        for node in nodes
    ]
    class_summary[circuit_class.value] = {
        "n_nodes": len(nodes),
        "mean_patch_effect": sum(effects) / len(effects),
        "nodes": [
            {
                "layer": node.layer,
                "head": node.head,
                "position": node.position.value,
                "patch_effect": effect,
            }
            for node, effect in zip(nodes, effects)
        ],
    }

for circuit_class in CircuitClass:
    summary = class_summary[circuit_class.value]
    print(
        f"{circuit_class.value:22s} n={summary['n_nodes']:2d} "
        f"mean Δ={summary['mean_patch_effect']:+.4f}"
    )

name_movers            n= 3 mean Δ=+0.1309
backup_name_movers     n= 8 mean Δ=-0.1417
negative_name_movers   n= 2 mean Δ=+0.9351
s_inhibition           n= 4 mean Δ=-0.7719
induction              n= 4 mean Δ=-0.2991
duplicate_token        n= 3 mean Δ=-0.3843
previous_token         n= 2 mean Δ=-0.1083


In [4]:
artifact = {
    "paper": "arXiv:2211.00593",
    "method": "published-circuit validation with semantic result patching",
    "config": {
        "model_name": "gpt2",
        "seed": SEED,
        "n_examples": N_DEV,
        "dataset_sha256": dataset.dataset_sha256,
        "semantic_positions": [position.value for position in SEMANTIC_POSITIONS],
        "activation": "result",
    },
    "baseline_logit_difference": baseline.item(),
    "direct_logit_attribution": dla.float().cpu().tolist(),
    "activation_patched_metric": patch_matrix.values.float().cpu().tolist(),
    "activation_patch_effect": patch_effect.float().cpu().tolist(),
    "published_classes": class_summary,
}
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")
artifact["config"]

Resultado salvo em results\paper\06_circuit_discovery.json


{'model_name': 'gpt2',
 'seed': 106,
 'n_examples': 4,
 'dataset_sha256': 'cb4f451c0de475c5e4180d0f9782ecc2e4d4f6bc08d424cf91a383dafe6e30a8',
 'semantic_positions': ['IO', 'S1', 'S2', 'S1+1', 'END'],
 'activation': 'result'}

## Leitura responsável

Os mapas são diagnósticos causais em um lote pequeno e não uma reexecução dos 100.000 exemplos do artigo. A presença dos heads nos plots apoia ou tensiona a lista publicada, mas não reclassificamos o circuito por um ranking ruidoso de quatro prompts. Em particular, a separação entre Name Movers primários e backups é analítica; a união preserva os 11 heads agrupados como `name mover` no código histórico.